- https://philliphaeusler.com/posts/aligning_tinystories/
    - https://github.com/pHaeusler/tinycatstories/tree/main

- 如何简单直观地理解 RL (on-policy REINFORCE 算法) 应用在 LLM 中
    - 不要淹没在 verl/openrlhf/trl 各种复杂的 rl4llm 的工程框架中，概念上（非实现上）做一次简单的祛魅吧
        - pg loss + kl reg + entropy reg：我们这个系列的第一期，verl loss components
    - 比如这里我们想从根本上改变 LLM 的行为，而不管提示如何。它应该总是谈论猫，即使被告知不要这样做。
    - 大模型可解释性分析中经常有通过 steering activation 来调控模型的行为（Anthropic golden gate 的例子），这一期内容，我们基于 rewards design 通过 online onpolicy 的RL的方式来调控模型的输出行为（simple align demo）；
- 如何在 LLM 自回归（rollout generation）的过程中计算联合概率
    - $\log p_\theta\!\big(y_{L+1:T}\mid y_{1:L}\big)
= \sum_{t=L+1}^{T} \log p_\theta\!\big(y_t \mid y_{1:t-1}\big).$
- 如何理解 loss
    - 负对数概率（negative log probabilities）乘以奖励（reward）的总和。
    - $\mathcal{L}(\theta) = -\mathbb{E}_{\tau \sim \pi_{\theta}} \left[ R(\tau) \frac{1}{T} \sum_{t=1}^{T} \log \pi_{\theta}(a_t | s_t) \right],$
        - $\mathcal{L}_{\text{REINFORCE}}(\theta) = -\sum_{t=1}^{T} G_t \log \pi_{\theta}(a_t | s_t) \quad \text{且} \quad G_t = R(\tau) \quad \forall t,$
            - 无 discount
            - 序列级单一回报
        - https://huggingface.co/learn/deep-rl-course/unit4/policy-gradient
    - 为何要引入 KL reg
        - $\mathcal{L}_{\text{total}}(\theta) = \underbrace{-R(\tau) \frac{1}{T} \sum_{t=1}^{T} \log \pi_{\theta}(a_t | s_t)}_{\text{REINFORCE}} + \underbrace{\beta \frac{1}{T} \sum_{t=1}^{T} D_{\text{KL}}(\pi_{\theta}(\cdot | s_t) \| \pi_{\text{ref}}(\cdot | s_t))}_{\text{逐步 KL 惩罚}}$
        - BV1rsAXe7EZs
        - kl 计算的细节（原作者写得有小问题）
        - https://github.com/volcengine/verl/blob/main/verl/workers/actor/dp_actor.py#L477-L480
- 对比理解 ppo4llm 的奖励与回报计算
    - 实现不会对“最终奖励”按 token 做指数折扣（常取 𝛾=1），而是把每个 token 当作一步，用逐 token 的 KL 代价做密集奖励，最终的序列奖励（来自 RM）只在最后一步加上；
    - $r_t = -\beta \text{KL}_t, \quad t=1, \dots, T-1; \quad r_T = R_{\text{RM}} - \beta \text{KL}_T,$
    - 基于 GAE 算各个token的优势：$A_t=G_t-V(s_t)$ ($\lambda =1$)
        - $G_t$（return to go）：总共拿了多少奖励（看到终点）。
        - $A_t$：比预期多拿了多少（相对基线/值函数）。
- online/offline, on-policy/off-policy
    - online: 训练时每个 epoch 都 用当前参数的策略直接采样新故事，然后立刻用回报计算梯度、更新参数；没有预先固定的数据集或回放缓存。代码里循环采样、计算 reward、反传更新是一气呵成的 online 交互流程
    - onpolicy: 轨迹是用当前策略 $\pi_\theta$ 逐 token 采样得到；损失里用的正是这些 token 的 $\log\pi_\theta(a_t|s_t)$ 之和；没有复用旧策略的样本、也没有做重要性采样修正。

In [1]:
from IPython.display import Image

In [2]:
import torch
from torch.distributions import Categorical
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import AdamW
import matplotlib.pyplot as plt

### TinyStories


> RL (最 simple 的 policy gradient) for align
> 
> 一些不拘一格的 reward function design
>
> KL loss 在语言能力和 reward max 的 tradeoff

- https://arxiv.org/abs/2305.07759
    - https://arxiv.org/abs/2305.07759
    - Ok, so tinystories is a fantastic paper that shows how **a small transformer model** can be trained to produce **coherent stories**.
        - TinyStories-33M
    - Their trick was to carefully **curate training data by synthetically generating it (using GPT)**. It worked!
- the perspective of RL
    - Rather than training on data (labeled or unlabeled), we can train with **another system that gives feedback**. This could be a simple function that evaluates the model state or action, perhaps from a simulator, or it could be a deep-learning model specifically trained to give feedback - Reinforcement Learning (RL).

### Embedding loss/rewards

In [2]:
#cannot import name 'DEFAULT_CIPHERS' from 'urllib3.util.ssl_' 
# !pip install --upgrade 'urllib3==1.26.7' 

In [3]:
# !pip install sentence_transformers

In [4]:
from sentence_transformers import SentenceTransformer, util

embedding_model = SentenceTransformer("all-MiniLM-L6-v2").to("cuda")
reference_embedding = embedding_model.encode("cat", convert_to_tensor=True)

def compute_rewards(sequences):
    sequence_embeddings = embedding_model.encode(sequences, convert_to_tensor=True)
    cosine_similarities = util.pytorch_cos_sim(
        reference_embedding.unsqueeze(0), sequence_embeddings
    ).squeeze()
    return cosine_similarities

In [5]:
reference_embedding.shape

torch.Size([384])

```
Sequence: Once upon a time there was a little cat., Cosine Similarity: 0.5711
Sequence: Once upon a time there was a little boy named, Cosine Similarity: 0.1272
Sequence: Once upon a time there was a very special little, Cosine Similarity: 0.1173
Sequence: Once upon a time there was a fat cat named, Cosine Similarity: 0.4979
Sequence: Once upon a time there was a furry cat., Cosine Similarity: 0.5783
mean cosine similarity: 0.48004984855651855
```

### REINFORCE

- `./scripts/reinforce_align.py`

- reward function (sentences 级别）
    $$
    R(s)=\text{cos}(\text{Emb(s),Emb('cat')})
    $$
- loss function （negative objective function)

    $$
    L(\theta)=L_{pg}(\theta)+\lambda D_{kl}(\theta)
    $$
    - $L_{\text{pg}}(\theta) = -\mathbb{E}_{s \sim \pi_\theta}[R(s) \cdot \log \pi_\theta(s)]$
        - The final loss is the sum of the negative log probabilities for the story multiplied by the reward.
        - The log probabilities represent how likely a generated sequence is according to the model. The more probable a sequence, the higher its log probability. This will be negative number. Taking the negative turns this value into a cost. We weight the cost by the reward and use back-propagation to minimize it.
    - $D_{\text{KL}}(\theta) = \mathbb{E}_{s \sim \pi_\theta}[D_{\text{KL}}(\pi_\theta(\cdot|s) || \pi_{\text{ref}}(\cdot|s))]$
- 采样及优化过程
    - 采样 batch_size 个序列（token by token，autoregressive）
        - $s_i \sim \pi_\theta, i \in \{1,...,N\}$
    - 计算每个序列的对数概率：
        - $\log \pi_\theta(s_i) = \frac1T\sum_{t=1}^T \log \pi_\theta(s_{i,t}|s_{i,<t})=\frac1T\log\Pi_{t=1}^T\pi_\theta(s_{i,t}|s_{i,\lt t})$
            - 整个句子的联合概率（joint distribution）的 log prob
            - To build up the joint probability of the generated story we accumulated the log probability of each selected token.
    - 计算 kl 散度（KL 散度约束保持生成文本的流畅性）
        - $D_{kl}=\frac1T\sum_{t=1}^TD_{kl}(\pi_{ref}(\cdot|s_{\lt t})\|\pi_\theta(\cdot|s_{\lt t}))$
        - foward kl：https://docs.pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html
            - RLHF/PPO 常用 reverse KL 惩罚；
        - 内部去算每个 token 位置的 kl 散度时，不应该除以 vocabulary size

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")

In [7]:
tokenizer.vocab_size

50257

In [3]:
Image(url='../figs/training_metrics_0.png', width=600)

In [2]:
Image(url='../figs/training_metrics_6000.png', width=600)